In [1]:
%pip install pyspark

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("OlistDataCleaning") \
    .config("spark.driver.host", "127.0.0.1") \
    .getOrCreate()

In [3]:
from pyspark.sql import functions as F
import csv
import os
import pandas as pd
from datetime import datetime

# Join reviews and orders/products on order_id
reviews_path = "reviews_final_20260331_223131.csv"
orders_products_path = "orders_and_products_final_20260331_231502.csv"

reviews_df = spark.read.csv(reviews_path, header=True, inferSchema=True)
orders_products_df = spark.read.csv(orders_products_path, header=True, inferSchema=True)

joined_df = orders_products_df.join(reviews_df, on="order_id", how="left")

# Show a quick sample
joined_df.show(5)

# Optional: write to disk
joined_df = joined_df.drop("order_id") 
joined_pd = joined_df.toPandas()
output_dir = os.getcwd()

output_path = os.path.join(
    output_dir, f"orders_and_reviews_final_{datetime.now():%Y%m%d_%H%M%S}.csv"
 )
joined_pd.to_csv(output_path, index=False, quoting=csv.QUOTE_ALL)
print(f"CSV salvo em: {output_path}")

+--------------------+--------------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+-------------+--------------------+-------------------+------------------+------------------+--------------+----------------+----------+------------+----------------------+
|            order_id|         customer_id|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|order_item_id|           seller_id|shipping_limit_date|             price|     freight_value| category_name|product_weight_g|volume_cm3|review_score|review_comment_message|
+--------------------+--------------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+-------------+--------------------+-------------------+------------------+------------------+--------------+---------------